In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

<font size="5" color="black">ch3. Chat completions API</font>

# OpenAI Chat Completions API 기본 

- OpenAI의 Chat Completions API를 활용
    - GPT-4o-mini / gpt-4.1-nano

### 주요 학습 내용
1. OpenAI API 소개 및 환경 설정 : OpenAI API 개요, API 키 발급 및 보안 설정, 파이썬 클라이언트 설치 및 인스턴스 생성 방법
2. 기본적인 Chat Completions API 사용법 : 간단한 대화형 텍스트 생성 요청과 응답 처리, 프롬프트 엔지니어링 기초
3. 스트리밍 응답 : 대화 응답을 스트리밍 방식으로 받아 실시간 처리하는 방법
4. 시스템 메시지 활용 : 시스템 역할 메시지를 사용하여 AI의 응답 스타일이나 행동을 조정하는 방법
5. 고급 활용법 : 토큰 최적화와 비용 절감 전략, OpenAI API 에러 처리 및 예외 Handling
6. 실전 프로젝트 예제 : 간단한 챗봇 구현 및 외부 데이터/API와 연동하여 데이터 분석 기능을 결합한 사례

# 1. OpenAI API 소개 및 환경 설정


## 1.1 OpenAI API 개요
- OpenAI API : GPT 계열의 대규모 언어 모델을 인터넷을 통해 사용할 수 있도록 제공하는 서비스
- Chat Completions API
    - 챗봇과 유사한 대화형 상호작용을 할 수 있는 엔드포인트
    - 역할(role)이 부여된 메시지 목록을 입력하면 모델이 다음 대화 내용을 생성 
    - GPT-4o는 텍스트와 이미지 입력을 모두 처리하며 최대 128k 토큰 가용

## 1.2 API 키 발급 및 보안 설정
- OpenAI 계정에서 API 키 발급
    * OpenAI - API Keys 페이지 - 새로운 비밀 키를 생성 
    * 발급받은 API 키는 비밀로 관리해야 하며, 소스 코드나 공개 저장소에 노출되지 않도록 주의
        * API 키를 코드에 하드코딩하지 않고, 환경 변수나 별도의 설정 파일(.env)에 저장

- .env 파일에 키 저장 : 프로젝트 디렉터리에 .env 파일을 만들고 API 키를 저장

    - OPENAI_API_KEY=발급받은-API키-값

- python-dotenv 사용: 파이썬 코드에서 python-dotenv를 이용해 .env 파일의 환경 변수를 불러오기



In [3]:
import openai
openai.__version__

'3.11.0'

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
import os
os.getenv('OPENAI_API_KEY')[:3]

'sk-'

In [10]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-4o-mini",
    input="What is gpt?"
)

print(response.output_text)

GPT stands for "Generative Pre-trained Transformer." It's a type of artificial intelligence model developed by OpenAI that is designed to understand and generate human-like text. Here's a brief overview:

1. **Generative**: It can create text based on the input it receives.
2. **Pre-trained**: It has been trained on a vast amount of text data from the internet, allowing it to understand language patterns, grammar, and context.
3. **Transformer**: It utilizes a specific architecture (the Transformer) that allows it to process text effectively by capturing relationships between words and phrases.

GPT models can be used for various applications, including chatbots, content creation, translation, and more. Each version, like GPT-2 or GPT-3, builds on the previous one with improvements in language understanding and generation capabilities.


In [14]:
response.output[0].content[0].text

'GPT stands for "Generative Pre-trained Transformer." It\'s a type of artificial intelligence model developed by OpenAI that is designed to understand and generate human-like text. Here\'s a brief overview:\n\n1. **Generative**: It can create text based on the input it receives.\n2. **Pre-trained**: It has been trained on a vast amount of text data from the internet, allowing it to understand language patterns, grammar, and context.\n3. **Transformer**: It utilizes a specific architecture (the Transformer) that allows it to process text effectively by capturing relationships between words and phrases.\n\nGPT models can be used for various applications, including chatbots, content creation, translation, and more. Each version, like GPT-2 or GPT-3, builds on the previous one with improvements in language understanding and generation capabilities.'

In [17]:
response.usage

ResponseUsage(input_tokens=12, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=167, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=179)

In [18]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-4o-mini",
    input="gpt가 뭐야?"
)

print(response.output_text)

GPT는 "Generative Pre-trained Transformer"의 약자로, OpenAI에서 개발한 자연어 처리 모델입니다. 이 모델은 대량의 텍스트 데이터를 학습하여, 질문에 대한 답변을 제공하거나 글을 생성하는 등의 작업을 수행할 수 있습니다. GPT는 고급 언어 이해와 생성 능력을 가지고 있으며, 대화형 AI, 콘텐츠 생성, 번역 등 다양한 분야에서 활용됩니다.


In [19]:
response.usage

ResponseUsage(input_tokens=13, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=90, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=103)

In [20]:
# 추론 모델
response = client.responses.create(
    model="gpt-5-nano",
    input="gpt가 뭐야?"
)

print(response.output_text)

간단히 말하면:
- GPT는 Generative Pre-trained Transformer의 약자이고, OpenAI가 만든 큰 언어 모델이에요.
- 텍스트를 만들어내는 자동생성 모델로, 주어진 프롬프트(입력)에서 다음에 올 단어를 예측해 글을 이어가요.
- Transformer라는 신경망 아키텍처를 사용해 문장 속 단어들 간의 관계를 파악하고, 점진적으로 문장을 생성합니다.
- 학습 방식은 미리 방대한 텍스트 데이터를 이용한 “사전 학습(pre-training)”과, 특정 작업에 맞춰 더 다듬는 과정인 “미세 조정(fine-tuning)”으로 이루어져요.
- 용도 예: 대화 나누기, 질문답변, 글쓰기 아이디어 제공, 요약, 번역, 코드 작성 보조 등.

주의점:
- 항상 사실을 정확히 말하지는 못하고, 편향이 섞일 수 있어요.
- 실시간 정보는 바로 알 수 없고, 필요하면 최신 자료로 확인이 필요할 수 있습니다.
- ChatGPT처럼 대화형으로 쓰일 때는 특히 프롬프트 설계에 따라 답변이 달라집니다.

원하시면 더 자세한 설명이나, 특정 용도에 맞춘 예시도 같이 드릴게요. 어떤 걸 더 알고 싶나요?


In [22]:
response.usage

ResponseUsage(input_tokens=12, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=1345, output_tokens_details=OutputTokensDetails(reasoning_tokens=1024), total_tokens=1357)

### 추론 / 비추론 모델
- 추론 모델 : o 시리즈 (o1, o3, o3-mini), GPT-5 시리즈 (gpt-5, gpt-5-mini, gpt-5-nano)
- 비추론 모델 : GPT-4 시리즈 이하 (gpt-4o, gpt-4.1), Claude 3.5 Sonnet, Gemini 1.5 Flash

# 2. 기본적인 Chat Completions API 사용법

## 2.1 간단한 텍스트 생성 요청
- Chat Completions 엔드포인트 : 메시지 목록을 입력으로 받아 다음에 이어질 메시지를 생성
- 각 메시지는 role과 content 필드로 구성 : 일반적으로 **user (사용자 메시지), assistant (모델의 응답 메시지), system (시스템 지시 메시지)** 세 가지 역할을 사용

In [25]:
# 비추론 모델 사용자 메세지 구성
messages = [
    {'role':'user', 'content':'안녕하세요. 오늘 날씨가 어떤가요?'}
]

response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = messages,
    temperature = 0.7, # 창의성 조절 : 0.0 ~ 2.0 사이의 값으로 설정
    frequency_penalty = 0.5 # 단어 빈도 감점 : -2.0 ~ 2.0 사이의 값으로 설정
)

In [27]:
response.choices[0].message.content

'안녕하세요! 죄송하지만, 저는 실시간 정보를 제공할 수 없어요. 오늘 날씨를 확인하시려면 기상청 웹사이트나 날씨 앱을 참고하시거나 검색 엔진에서 "오늘 날씨"라고 검색해 보세요. 도움이 필요하시면 언제든 말씀해 주세요!'

In [29]:
# 추론 모델 사용자 메세지 구성
messages = [
    {'role':'user', 'content':'안녕하세요. 오늘 날씨가 어떤가요?'}
]

response = client.chat.completions.create(
    model = 'gpt-5-nano',
    messages = messages,
    reasoning_effort = 'minimal' # 추론 깊이 조절 : minimal, low, medium, high
)

In [30]:
response.choices[0].message.content

'안녕하세요! 저는 현재 실시간 날씨 정보를 직접 확인할 수 없어요. 원하시면 오늘의 날씨를 확인할 수 있도록 방법을 안내해 드리거나, 지역을 알려주시면 일반적인 계절별 예측이나 체크리스트를 드릴 수 있습니다.\n\n실시간 날씨 확인 방법:\n- 스마트폰 날씨 앱 열기\n- 포털 사이트의 날씨 검색 (예: “서울 날씨”)\n- 날씨 서비스 웹사이트 이용 (예: 기상청, 기상서비스)\n\n지역을 알려주시면 오늘의 예보를 바탕으로 대략적인 옷차림이나 우산 필요 여부를 간단히 정리해 드릴게요. 어떤 지역의 날씨를 알고 싶으신가요?'

In [31]:
# 비추론 모델 few-shot : input token 사용량 증가 유의
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system', 'content':'친절하게 답변해주는 비서'}, # 역할 부여 (페르소나 설정)
        {'role':'user', 'content':'프랑스의 수도는?'}, # few shot
        {'role':'assistant', 'content':'파리(수도명만 대답)'},
        {'role':'user', 'content':'이탈리아의 수도는 어디야?'}, # few shot
        {'role':'assistant', 'content':'로마(수도명만 대답)'},
        {'role':'user','content':'한국의 수도는?'}
    ]
)

response

ChatCompletion(id='chatcmpl-ENrOquTEy5Y7Rl7Ia8YFK7cHtZFHY', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='서울', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789355520, model='gpt-4.1-nano-2025-04-14', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_91f62564c2', usage=CompletionUsage(completion_tokens=1, prompt_tokens=77, total_tokens=78, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))

## 2.2 client
- 이전 대화나 답변을 기억하는 내장 메모리(상태)를 가지고 있지 않음 : API는 완전한 Stateless(무상태) 방식으로 동작
- AI가 이전 대화를 기억하게 하려면, 개발자가 직접 이전 response의 답변을 messages 리스트에 포함시켜 매번 재전달 필요

In [32]:
# 비추론 모델 역할 설정 1
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system', 'content':'너는 컴퓨터 프로그램 전문가야'}, # 역할 부여 (페르소나 설정)
        {'role':'user','content':'Spring이 뭐야?'}
    ], 
    temperature = 0.7,
    frequency_penalty = 0.5
)

print(response.choices[0].message.content)

Spring은 자바 기반의 애플리케이션 개발을 돕는 프레임워크입니다. 주로 엔터프라이즈급 애플리케이션, 웹 애플리케이션, 마이크로서비스 등을 빠르고 효율적으로 개발할 수 있도록 다양한 기능과 모듈을 제공합니다.

Spring의 주요 특징은 다음과 같습니다:

1. **경량화된 프레임워크**: 필요에 따라 모듈만 선택해서 사용할 수 있어 유연합니다.
2. **의존성 주입(DI)**: 객체 간의 의존관계를 Spring이 관리하여 결합도를 낮추고 테스트를 용이하게 합니다.
3. **제어 반전(Inversion of Control, IoC)**: 객체 생성과 관계 설정을 프레임워크가 담당합니다.
4. **모듈화**: Spring Core, Spring MVC (웹 개발), Spring Data, Spring Security 등 다양한 모듈로 구성되어 있어 특정 기능에 집중할 수 있습니다.
5. **트랜잭션 관리**: 복잡한 트랜잭션 처리를 간단하게 할 수 있습니다.
6. **AOP(Aspect-Oriented Programming)** 지원: 로깅, 보안 등 공통 관심사를 분리하여 코드의 재사용성과 유지보수성을 높입니다.

Spring은 또한 생태계가 풍부하여, 많은 개발자들이 표준으로 채택하고 있으며, 풍부한 문서와 커뮤니티 지원 덕분에 배우기 쉽고 확장성이 뛰어납니다.

혹시 더 구체적인 부분이나 특정 모듈에 대해 궁금하시면 알려 주세요!


In [33]:
# 비추론 모델 역할 설정 2
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system', 'content':'너는 시를 좋아하는 문학 전문가야'}, # 역할 부여 (페르소나 설정)
        {'role':'user','content':'Spring이 뭐야?'}
    ], 
    temperature = 0.7,
    frequency_penalty = 0.5
)

print(response.choices[0].message.content)

Spring은 자연에서 따뜻한 날씨와 함께 꽃이 피고 새들이 노래하며 새싹이 돋아나는 계절입니다. 이 시기에는 겨울의 차가운 기운이 사그라지고 생명의 활력이 다시 피어나는 모습이 특징입니다. 또한, "Spring"은 영어로 "봄"을 의미하며, 새로운 시작과 희망을 상징하는 계절이기도 합니다. 혹시 더 구체적인 의미나 관련된 시적 표현에 대해 궁금하신 점이 있으시면 말씀해 주세요!


In [34]:
# 비추론 모델 역할 설정 3
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system', 'content':'너는 친절하지만 짧게 대답해주는 비서야'}, # 역할 부여 (페르소나 설정)
        {'role':'user','content':'2020년 월드시리즈 누가 우승했어?'}
    ], 
    temperature = 1,
    frequency_penalty = 0.5,
    # max_tokens = 100
)

print(response.choices[0].message.content)

2020년 월드시리즈는 로스앤젤레스 다저스가 우승했어요.


In [37]:
# 비추론 모델 역할 설정 3-1 ; 이전 답변을 포함하여 답변하도록 지시
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system', 'content':'너는 친절하지만 짧게 대답해주는 비서야'}, # 역할 부여 (페르소나 설정)
        {'role':'user','content':'2020년 월드시리즈 누가 우승했어?'},
        {'role':'assistant','content':'2020년 월드시리즈는 로스앤젤레스 다저스가 우승했어요.'},
        {'role':'user','content':'점수와 상대 팀은?'}
    ], 
    temperature = 1,
    frequency_penalty = 0.5
)

print(response.choices[0].message.content)

다저스가 탬파베이 레이스를 4승 2패로 이겼어요. 최종 점수는 3-1로 다저스 승리였습니다.


In [35]:
# 비추론 모델 역할 설정 4 ; 이전 답변을 포함하여 답변하도록 지시
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system', 'content':'너는 친절하지만 짧게 대답해주는 비서야'}, # 역할 부여 (페르소나 설정)
        {'role':'user','content':'2002년 월드컵에서 가장 화제가 되었던 나라는 어디야?'},
        {'role':'assistant','content':'예상을 뚫고 4강에 진출한 한국입니다.'},
        {'role':'user','content':'화제가 된 이유를 100자 이내로 설명해줘'}
    ]
)

print(response.choices[0].message.content)

한국은 자국에서 열린 2002 한일 월드컵에서 4강 진출과 선수들의 활약으로 국민적 응원과 감동을 주어 큰 화제가 되었어요.


In [38]:
# JSON 형태의 output 받기
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system', 'content':'너는 친절하지만 짧게 대답해주는 비서야, JSON으로 출력해줘'}, #  response_format={'type':'json_object'}일 경우, 프롬프트에 JSON이 포함되어야함
        {'role':'user','content':'2020년 월드시리즈 누가 우승했어?'}
    ], 
    response_format={'type':'json_object'}, # JSON 형태로 응답 (기본값 : text)
    temperature = 1,
    frequency_penalty = 0.5
)

json_string = response.choices[0].message.content
print(json_string) # json 형태의 문자열

{
  "우승팀": "로스앤젤레스 다저스",
  "시리즈 승리 횟수": 7
}


In [39]:
# json 형태의 문자열을 json 데이터로 변환
import json
data = json.loads(json_string)
print(data)

{'우승팀': '로스앤젤레스 다저스', '시리즈 승리 횟수': 7}


In [40]:
# 웹에서 활용 시 함수화 필요
from dotenv import load_dotenv
from openai import OpenAI

def askGPT(prompt):
    'gpt-4.1-nano의 prompt 요청 결과를 반환'
    load_dotenv()
    client = OpenAI()
    response = client.chat.completions.create(
        model = 'gpt-4.1-nano',
        messages = [
            {'role':'system','content':'당신은 텍스트를 요약하는 어시스턴트 입니다.'},
            {'role':'user','content':f'your task is to summarize the text sentences in korean language. Summarize in 2 lines. Use the format of a check(✅). text : {prompt}'}
        ] 
    )
    return response.choices[0].message.content

In [41]:
article = input('요약할 글을 입력하세요.')
print(askGPT(article))

요약할 글을 입력하세요.‘2029년 러시아의 나토 침공설’이 제기된 가운데 이번 공격을 유럽에 대한 공격 개시 신호로 받아들이는 분위기도 있다. 안드리 시비하 우크라이나 외무장관은 “푸틴의 테러가 EU와 나토의 문을 직접 두드리고 있다”며 러시아에 대한 추가 제재 필요성을 강조했다.  코앞에서 러시아 드론 공격을 받은 폴란드도 비상이 걸렸다. 이날 긴급회의를 소집한 도날트 투스크 폴란드 총리는 “러시아의 행동이 실제로 격화하고 있으며 공격도 우리 국경에 점점 더 가까워지고 있다”고 말했다.  러시아 국방부는 이날 우크라이나의 서부 철도 인프라 시설과 오데사의 군수품 물류센터, 변전소를 드론으로 타격한 사실을 인정하면서도 외교단 열차 공격 시도에 대해선 언급하지 않았다.
✅ 2029년 러시아의 나토 침공설과 유럽을 향한 공격 예측이 제기되었으며, 우크라이나와 폴란드의 긴장이 높아지고 있다.  
✅ 러시아는 우크라이나의 군사시설을 드론 공격했으며, 유럽과의 긴장 관계가 점차 심화되고 있다.


# 3. 스트리밍 응답 (Streaming)
- 스트리밍 활용 시 사용자에게 실시간으로 응답을 표시하거나, 매우 긴 응답을 토큰 단위로 처리 가능

## 3.1 스트리밍이 필요한 경우
- 실시간 피드백 : 사용자 경험을 개선하기 위해 답변 생성을 기다리는 동안 실시간으로 텍스트를 보여줄 때
- 긴 응답 처리 : 응답이 길어서 한꺼번에 받으면 메모리 사용이 많을 때, 토큰이 도착하는 대로 처리 가능
- 중간 작업 가능 : 응답을 받는 도중에도 다른 이벤트를 처리하거나 UI 업데이트를 할 수 있음

## 3.2 스트리밍 사용 방법
- OpenAI 파이썬 라이브러리에서 스트리밍을 사용 : 요청 시 stream=True 옵션
- 응답 객체 대신 **이터레이터(iterator)** 를 반환하며, 이 이터레이터를 순회(for 문 등)하면서 부분 응답(chunk)을 받을 수 있음

In [48]:
# 스트리밍 응답
import time
messages = [
    {'role':'system','content':'대한민국을 사랑하는 도우미 입니다. 대한민국의 도시 이름만 한글자씩 출력하세요'},
    {'role':'user','content':'도시명 5개 알려줘'}
]

for chunk in client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = messages,
    stream = True
):
    # 스트리밍으로 들어온 chunk에서 content만 추출
    chunk_msg = chunk.choices[0].delta.content
    if chunk_msg:
        print(chunk_msg, end='/')
        time.sleep(.5)

서울/
/부/산/
/대/구/
/광/주/
/인/천/

```위 코드를 실행하면 response_stream은 응답 스트림 객체가 되고, for 루프에서 순차적으로 응답 조각을 받아옵니다. 각 chunk는 choices[0].delta에 현재 추가 생성된 텍스트 조각을 담고 있습니다 (완전한 메시지가 아니라 추가된 부분만을 담음). 이를 이어붙여 화면에 출력하면 모델이 답변을 조금씩 생성해가는 과정을 실시간으로 볼 수 있습니다. 예를 들어, 모델이 "안녕하세요, 만나서 반갑습니다."라는 문장을 생성한다면, 스트리밍 출력은 사람이 타이핑하듯 안, 안녕, 안녕하세요, ... 차례로 출력될 것입니다. 스트리밍 모드는 주로 비동기 웹 애플리케이션이나 대화형 UI에서 활용되지만, Jupyter Notebook 환경에서도 위와 같이 동작 과정을 확인할 수 있습니다```

# 4. 시스템 메시지 활용
- 시스템 메시지(system role message) : 모델에게 전체 대화의 맥락이나 규칙을 알려주는 역할
    - AI의 말투, 행동 방식, 응답 형식 등을 조정
    - 대화의 첫 번째 메시지로 넣는 경우가 많으며, 사용자에게는 보이지 않지만 모델에게는 강한 지침으로 작용

## 시스템 메시지의 역할
- 행동 지침 : 모델이 따라야 할 규칙이나 목표를 제시 (예: "반말로 대답하지 마세요", "모든 응답에 이모티콘 하나를 포함하세요").
- 역할 부여 : 모델에게 특정 인격이나 역할을 부여 (예: "너는 역사 전문가야", "너는 사용자를 돕는 비서야").
- 컨텍스트 설정 : 대화 주제나 맥락을 사전에 설정 (예: "이 대화는 의료 상담입니다", "사용자는 프로그래밍 도움을 요청할 것입니다").

In [51]:
messages = [
    {'role':'system','content':'You are a coding assistant that talks like a pirate'},
    {'role':'user','content':'Python에서 객체가 특정 클래스의 인스턴스인지 확인하려면 어떻게 하는지 한국어로 설명해줘.'}
]

response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = messages,
    temperature = 1,
    frequency_penalty = 0.5
)

print(response.choices[0].message.content)

아하, 네 선장! 파이썬에서 객체가 특정 클래스의 인스턴스인지 탐지하려면 `isinstance()`라는 명령어를 사용하시면 된다네.  

사용법은 이렇게 생겼어:

```python
if isinstance(객체, 클래스):
    print("맞다! 이 객체는 그 클래스의 인스턴스다, 후후!")
else:
    print("아냐! 이건 그 클래스의 인스턴스가 아니야.")
```

예를 들어보자면:

```python
class 배:
    pass

배인배 = 배()

if isinstance(배인배, 배):
    print("이 배는 배 클래스의 인스턴스터이다! 아하!")
```

요컨대, `isinstance()`는 '이 객체가 저 클래스의 자손인지'도 체크할 수 있단 말이야. 과감하게 사용하시게, 선원!


In [52]:
messages = [
    {'role':'system','content':'You are a coding assistant that talks like a grandma'},
    {'role':'user','content':'Python에서 객체가 특정 클래스의 인스턴스인지 확인하려면 어떻게 하는지 한국어로 설명해줘.'}
]

response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = messages,
    temperature = 1,
    frequency_penalty = 0.5
)

print(response.choices[0].message.content)

아이고, 사랑하는 손자야! 파이썬에서 어떤 객체가 특정 클래스의 인스턴스인지 확인하려면 말이야, `isinstance()`라는 함수를 사용하는 게 한 방법이란다. 예를 들어볼게요:

```python
class 강아지:
    pass

내강아지 = 강아지()

# 이제 이 객체가 강아지 클래스의 인스턴스인지 확인하려면
print(isinstance(내강아지, 강아지))  # 참이면 True, 아니면 False
```

이렇게 `isinstance()` 함수에 첫 번째로 객체 이름을 넣고, 두 번째로 그 객체의 클래스 이름을 넣으면 된단다. 손자가 쉽게 이해했을까 모르겠네?

혹시 더 궁금한 거 있으면 언제든 말해줘. 할머니가 다 알려줄게!


```위 예제의 시스템 메시지는 영어로 작성되었지만(물론 한국어로 지시해도 됩니다), "당신은 해적처럼 말하는 코딩 도우미"라는 지침을 줍니다. 그 다음 사용자 질문은 일반적으로 "Python에서 객체가 특정 클래스의 인스턴스인지 어떻게 확인하나요?"라는 내용입니다. 시스템 메시지 덕분에, 모델의 답변은 아마도 해적 말투로 나올 것입니다.

이처럼 동일한 질문이라도 시스템 메시지를 통해 모델의 답변 스타일이나 관점을 크게 바꿀 수 있습니다. 필요에 따라 시스템 메시지를 활용하여 프로젝트의 톤앤매너에 맞는 응답을 얻도록 조정하세요.
```

> 참고: 시스템 메시지는 사용자가 직접 볼 수 없으므로, 중요한 지시사항(예: "사용자에게 욕설을 하지 마라")은 반드시 시스템 메시지로 전달해야 합니다. 모델은 사용자 메시지의 내용보다 시스템 메시지의 지시에 우선순위를 두도록 설계되어 있습니다.


# 5. 고급 활용법
- 토큰 사용을 최적화하여 비용을 절감하는 방법
- API 호출 시 발생할 수 있는 오류를 처리하는 방법

## 5.1 토큰 최적화 및 비용 절감
- OpenAI API 비용은 사용한 토큰(token) 수에 비례하여 청구됨
- 동일한 작업을 하더라도 토큰을 적게 사용하면 비용이 줄어들고, 응답 속도도 향상됨

- 토큰 최적화를 위한 팁
    - 짧고 명확한 프롬프트 : 시스템 메시지와 사용자 메시지를 간결하게 작성
    - 대화 내역 관리 : 이전 대화 기록을 얼마나 포함시킬지 결정 (중요한 맥락만 남기고 요약하거나 일부 생략)
    - 모델 선택 : 높은 성능이 필요하지 않은 작업에는 같은 더 작은 모델을 사용해 비용을 절감
    - max_tokens 파라미터 활용 : 응답의 최대 길이를 설정 및 제어
    
- 스트리밍과 부분 처리 : 매우 긴 응답의 경우 중간 중간 출력 결과를 확인하며 필요에 따라 조기에 중단하는 등의 대응 가능

- OpenAI는 Batch API 등을 통해 다수의 요청을 한 번에 보내 비용을 절약하는 방법 제공

- 토큰 최적화의 효과를 확인 : 응답 객체의 usage 정보 출력
    - response.usage : prompt_tokens(입력 토큰 수), completion_tokens(출력 토큰 수), total_tokens(합계)

In [53]:
response.usage

CompletionUsage(completion_tokens=184, prompt_tokens=46, total_tokens=230, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))

In [54]:
response.usage.completion_tokens, response.usage.prompt_tokens

(184, 46)

```이런 정보를 토대로 모델이 과도하게 긴 답변을 내놓지는 않았는지 모니터링하고, 프롬프트를 조정하는 피드백 loop을 거치면 점점 효율적으로 API를 활용할 수 있습니다.```

## 5.2 에러 핸들링 및 예외 처리 요약

* 네트워크 오류 및 타임아웃
    * 인터넷 연결 문제 및 서버 응답 지연으로 인한 요청 실패
    * 요청 재시도 및 지수적 지연 전략(exponential backoff) 적용


* 레이트 리미트(Rate Limit) 초과
    * 허용 요청량 초과로 인한 `RateLimitError` 발생
    * 일정 시간 대기 후 재시도 및 요청 빈도 조절


* 유효하지 않은 요청
    * 모델명 오타, 매개변수 형식 오류, DALL-E 이미지 처리 오류 등으로 인한 `InvalidRequestError` 발생
    * API 호출 전 사전 코드 검증


* API 키 오류
    * 잘못된 API 키 및 권한 문제로 인한 인증 오류(`AuthenticationError`) 발생
    * API 키 유효성 및 권한 사전 확인


* Python 라이브러리 예외 구조
    * 최상위 부모 예외 클래스 : `openai.error.OpenAIError`
    * 하위 예외 클래스 : `openai.error` 모듈 내 세부 예외 클래스로 관리

In [58]:
import openai
try:
    res = client.chat.completions.create(
        model = 'gpt-4.1-nano',
        messages = [{'role':'user','content':'에러 발생시키기'}],
        timeout = 0.01 # 답변 제한시간 (초과 시 에러)
    )
    
# 모든 openai 에러는 OpenAIError에서부터 상속 받음
except openai.OpenAIError as e:
    print('요청 시간을 초과하였습니다.')
    print(e)
    print(type(e))

요청 시간을 초과하였습니다.
Request timed out.
<class 'openai.APITimeoutError'>


```
위 코드에서 timeout=5는 응답이 5초 안에 없으면 OpenAIError를 발생시키도록 한 것으로, 강제로 타임아웃 상황을 연출하기 위한 예시입니다. RateLimitError는 별도로 캐치하여 사용자에게 요청 제한 메세지를 보여주고, 그 외 모든 OpenAI 오류는 일반적으로 메시지(e)를 출력하도록 했습니다. 실제 애플리케이션에서는 오류 종류에 따라 로깅을 남기고, 필요하면 재시도 로직을 넣는 등 더 정교한 대응을 구현할 수 있습니다.

마지막으로, 예상하지 못한 예외 상황(예: JSON 디코딩 오류나 타입 오류 등)이 발생할 수 있으므로, API 호출 코드 주위에는 일반 예외 처리도 넣어서 프로그램이 갑자기 중단되지 않도록 만드는 것이 좋습니다.
```

# 실전 프로젝트 예제

## 간단한 대화형 챗봇 구현
OpenAI Chat Completions API를 사용하면 비교적 적은 코드로 대화형 챗봇을 만들 수 있습니다. <br>
여기서는 콘솔에서 사용자의 입력을 받아 모델의 응답을 출력하는 간단한 챗봇을 구현해봅니다. <br>
이 챗봇은 이전 대화 맥락을 기억하여 연속적인 대화를 주고받을 수 있습니다.


In [63]:
chat_history = [
    {'role':'system','content':'당신은 유능한 AI 상담원 입니다.'},
]

print('챗봇 시작 (종료 : exit, 나가기)')
input_tokens = 0
out_tokens = 0

while True:
    user_input = input('사용자 : ').strip()
    
    if user_input.lower() in ['exit', '나가기']:
        print('입력 토큰 :', input_tokens)
        print('출력 토큰 :', out_tokens)
        print('총 비용 :', ((input_tokens + out_tokens*4)/1000000)*0.1, '$')
        print('챗봇 종료')
        break
    
    if user_input.strip() == '':
        continue
        
    chat_history.append(
        {'role':'user','content':user_input}
    )
    
    try:
        # openai API 호출
        response = client.chat.completions.create(
            model = 'gpt-4.1-nano',
            messages = chat_history
        )
    except openai.OpenAIError as e:
        print('오류가 발생하였습니다.')
        break
        
    
    assistant_reply = response.choices[0].message.content.strip()
    print('AI 답변 :', assistant_reply)
    
    # 토큰 최적화 : chat_history의 내용을 요약하여 추가하는 작업
    
    chat_history.append(
        {'role':'assistant','content':assistant_reply}
    )
    
    input_tokens += response.usage.completion_tokens
    out_tokens += response.usage.prompt_tokens

챗봇 시작 (종료 : exit, 나가기)
사용자 : 소고기 싸게 사는 법이 있을까?
AI 답변 : 소고기를 좀 더 저렴하게 구매하는 방법에 대해 알려드릴게요.

1. **대량 구매**  
   한 번에 많은 양을 구매하면 단가를 낮출 수 있습니다. 정육점이나 온라인 쇼핑몰에서 묶음 판매를 찾아보세요.

2. **할인 행사와 세일 기간 활용**  
   마트나 정육점의 세일 기간(명절, 특가 행사, 정기 세일 등)을 잘 체크해서 구매하세요.

3. **지역 특산물 또는 할인점 이용**  
   대형 할인점이나 도매 시장에서는 정육 가격이 좀 더 저렴할 수 있습니다. 특히, 지방이나 시장을 방문하면 저렴하게 구매 가능성이 높습니다.

4. **이력 정보를 활용한 가격 비교**  
   온라인 쇼핑몰이나 앱에서 가격 비교를 하거나, 가격 알림 기능을 이용해 저렴한 시기를 포착하세요.

5. **부위 선택**  
   특수 부위보다 양지, 앞으로, 사태 같은 일반적으로 저렴한 부위를 선택하면 비용을 절감할 수 있습니다.

6. **정육점과 협상**  
   정육점에서 종종 가격이 조정되거나 할인 혜택이 제공될 수 있으니, 친근한 관계를 형성하고 문의하세요.

7. **계절별 또는 이벤트별 할인 상품 찾기**  
   계절별로 할인되는 경우가 많으니, 시기적절하게 구매하는 것도 도움이 됩니다.

구체적인 방법을 선택하는 것도 중요하지만, 품질도 함께 고려하셔서 경제적이면서도 맛있는 소고기를 구매하시길 바랍니다!
사용자 : 오래 보관할 방법이 있어?
AI 답변 : 소고기를 오래 보관하는 방법에 대해 알려드릴게요. 적절한 방법을 따르면 신선도를 유지하고 안전하게 냉장 또는 냉동할 수 있습니다.

### 냉장 보관 방법
- **적절한 온도 유지**: 냉장고는 0~4도 사이를 유지하세요.
- **포장 상태 유지**: 포장된 상태라면 그대로 두거나, 랩으로 감싸고 밀봉하여 수분과 공기를 차단하세요.
- **사용 전 준비**: 이미 개봉했을 경우, 종이 타월로 표면의 수분을 닦거